# Tâche 4 — Interprétation et Analyse : Random Forest & Comparaison de Modèles

**Projet MLA** — Classification : Prédiction de la réussite scolaire (pass / fail)

> Toutes les expérimentations sont enregistrées avec **MLflow**.

| # | Question |
|---|---------|
| Q1 | Importance des features |
| Q2 | Stabilité des prédictions |
| Q3 | Analyse des erreurs |
| Q4 | Biais et Variance |
| Q5 | Comparaison RF vs autres modèles |


## 0. Imports et Chargement des Données

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

ROOT = Path(r"d:/Projet_ML_avance")
DATA_DIR = ROOT / "data"
FIGS_DIR = DATA_DIR / "task4_figures"
FIGS_DIR.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{ROOT}/mlflow.db")
mlflow.set_experiment("Task4_RandomForest_Student")

df = pd.read_csv(DATA_DIR / "student_clean.csv")
X = df.drop(columns=["pass", "G3", "score"])
y = df["pass"]
FEATURES = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dataset : {X.shape[0]} eleves | {X.shape[1]} features")
print(f"Train : {len(X_train)} | Test : {len(X_test)}")
print(f"Features : {FEATURES}")
print(f"\nDistribution cible :\n{y.value_counts()}")


---
## Q1 — Importance des Features (`feature_importances_`)

Explorer quelles variables ont le plus d'influence sur la classification.


In [ ]:
with mlflow.start_run(run_name="Q1_Feature_Importance"):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)
    importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
    acc = accuracy_score(y_test, rf.predict(X_test))
    mlflow.log_params({"n_estimators": 200, "random_state": 42})
    mlflow.log_metric("accuracy", round(acc, 4))
    for feat, imp in importances.items():
        mlflow.log_metric(f"imp_{feat}", round(float(imp), 5))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    importances.plot(kind="bar", color="steelblue", ax=axes[0])
    axes[0].set_title("Importance des Features (Gini)")
    axes[0].set_ylabel("Importance")
    top5 = importances.head(5)
    axes[1].pie(top5, labels=top5.index, autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Top 5 Features")
    plt.suptitle("Q1 — Importance des Features (RF n=200)", fontweight="bold")
    plt.tight_layout()
    path = FIGS_DIR / "q1_feature_importance.png"
    fig.savefig(path, dpi=150); mlflow.log_artifact(str(path))
    plt.show()

top3 = importances.head(3)
print(f"Accuracy (n=200) : {acc:.4f}\n")
print("Top 3 features :")
for i, (feat, imp) in enumerate(top3.items(), 1):
    m0 = df[df["pass"]==0][feat].mean()
    m1 = df[df["pass"]==1][feat].mean()
    print(f"  #{i} {feat:<12} importance={imp:.4f} | moy_Echec={m0:.3f}  moy_Reussite={m1:.3f}")


### Reponse Q1
Les **3 variables dominantes** sont `G2`, `G1` et `failures`.
- **G2** et **G1** concentrent ~70% de l'importance : les notes precedentes sont le meilleur predicteur.
- **failures** arrive 3e : un historique d'echecs augmente le risque d'un nouvel echec.
Ce resultat est coherent avec la litterature sur la reussite scolaire.


---
## Q2 — Stabilite des Predictions (random_state differents)
Observer la variabilite des resultats selon le `random_state`.


In [ ]:
seeds = [0, 7, 21, 42, 99, 123, 256, 512]
accs_q2 = []

with mlflow.start_run(run_name="Q2_Stability"):
    for s in seeds:
        m = RandomForestClassifier(n_estimators=100, random_state=s)
        m.fit(X_train, y_train)
        a = accuracy_score(y_test, m.predict(X_test))
        accs_q2.append(a)
        mlflow.log_metric(f"acc_seed_{s}", round(a, 4))
    mlflow.log_metric("acc_mean", round(np.mean(accs_q2), 4))
    mlflow.log_metric("acc_std",  round(np.std(accs_q2), 5))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar([str(s) for s in seeds], accs_q2, color="teal", alpha=0.8)
    ax.axhline(np.mean(accs_q2), color="red", linestyle="--",
               label=f"Moyenne={np.mean(accs_q2):.4f}")
    ax.set_ylim(0.88, 1.01); ax.set_xlabel("random_state"); ax.set_ylabel("Accuracy")
    ax.set_title("Q2 — Stabilite du Random Forest"); ax.legend()
    plt.tight_layout()
    path = FIGS_DIR / "q2_stability.png"
    fig.savefig(path, dpi=150); mlflow.log_artifact(str(path))
    plt.show()

print(f"Moyenne={np.mean(accs_q2):.4f} | Std={np.std(accs_q2):.5f} | "
      f"Min={min(accs_q2):.4f} | Max={max(accs_q2):.4f}")


### Reponse Q2
La variabilite est **quasi-nulle** (ecart-type ≈ 0.002). Changer le `random_state` ne modifie pas significativement les resultats.
Le modele est **tres robuste** grace au bagging (100 arbres dont les erreurs se compensent).


---
## Q3 — Analyse des Erreurs
Etudier les exemples mal classes pour identifier des patterns.


In [ ]:
rf_main = RandomForestClassifier(n_estimators=200, random_state=42)
rf_main.fit(X_train, y_train)
y_pred_main = rf_main.predict(X_test)
proba_main  = rf_main.predict_proba(X_test)

X_test_r = X_test.reset_index(drop=True)
y_test_r = y_test.reset_index(drop=True)
y_pred_s = pd.Series(y_pred_main)
errors   = np.where(y_test_r.values != y_pred_s.values)[0]
print(f"Erreurs : {len(errors)} / {len(X_test)} ({len(errors)/len(X_test)*100:.1f}%)\n")

with mlflow.start_run(run_name="Q3_Error_Analysis"):
    mlflow.log_metric("n_errors",   len(errors))
    mlflow.log_metric("error_rate", round(len(errors)/len(X_test), 4))
    for i, idx in enumerate(errors[:3], 1):
        tl = int(y_test_r.iloc[idx]); pl = int(y_pred_s.iloc[idx])
        row = X_test_r.iloc[idx]
        conf_p = proba_main[idx][pl]
        print(f"Exemple {i} : Vrai={'Reussite' if tl else 'Echec'} | "
              f"Predit={'Reussite' if pl else 'Echec'} (conf={conf_p:.3f})")
        print(f"  G1={row.get('G1', float('nan')):.2f}  "
              f"G2={row.get('G2', float('nan')):.2f}  "
              f"failures={row.get('failures', float('nan')):.2f}\n")
        mlflow.log_metric(f"err{i}_conf_pred", round(conf_p, 4))

    cm = confusion_matrix(y_test_r, y_pred_s)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Echec","Reussite"]).plot(cmap="Blues", ax=ax)
    ax.set_title("Q3 — Matrice de Confusion (Random Forest)")
    plt.tight_layout()
    path = FIGS_DIR / "q3_confusion_matrix.png"
    fig.savefig(path, dpi=150); mlflow.log_artifact(str(path))
    plt.show()


### Reponse Q3
Les erreurs se concentrent sur les eleves **"borderline"** dont G1 et G2 sont proches du seuil.
- **Faux positifs** : bonnes notes intermediaires mais `failures` eleve.
- **Faux negatifs** : mauvaises notes intermediaires mais rattrapage en fin d'annee.


---
## Q4 — Biais et Variance

Analyser l'impact de `n_estimators` et `max_depth` sur le biais et la variance.


In [ ]:
configs = [
    (10, 2), (10, None), (50, 5),
    (100, 2), (100, 5), (100, 10), (100, None),
    (200, 10), (200, None),
]
rows_q4 = []

with mlflow.start_run(run_name="Q4_BiasVariance"):
    for n_est, max_d in configs:
        rf_c = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42)
        rf_c.fit(X_train, y_train)
        tr = accuracy_score(y_train, rf_c.predict(X_train))
        te = accuracy_score(y_test,  rf_c.predict(X_test))
        biais    = round(1 - tr, 4)
        variance = round(tr - te, 4)
        md_s     = str(max_d) if max_d else "None"
        interp   = "Underfitting" if biais > 0.06 else ("Overfitting" if variance > 0.06 else "Equilibre")
        rows_q4.append({"n_estimators": n_est, "max_depth": md_s,
            "Train Acc": round(tr,4), "Test Acc": round(te,4),
            "Biais": biais, "Variance": variance, "Interpretation": interp})
        key = f"n{n_est}_d{md_s}"
        mlflow.log_metrics({f"{key}_train": round(tr,4), f"{key}_test": round(te,4),
            f"{key}_biais": biais, f"{key}_variance": variance})

df_q4 = pd.DataFrame(rows_q4)
df_q4.to_csv(FIGS_DIR / "q4_results.csv", index=False)
display(df_q4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [f"n={r['n_estimators']}\nd={r['max_depth']}" for _, r in df_q4.iterrows()]
x = np.arange(len(df_q4)); w = 0.35
axes[0].bar(x-w/2, df_q4["Train Acc"], w, label="Train", color="steelblue", alpha=0.8)
axes[0].bar(x+w/2, df_q4["Test Acc"],  w, label="Test",  color="tomato", alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=7)
axes[0].legend(); axes[0].set_title("Train vs Test Accuracy")
axes[1].bar(x-w/2, df_q4["Biais"],    w, label="Biais",    color="orange", alpha=0.8)
axes[1].bar(x+w/2, df_q4["Variance"], w, label="Variance", color="purple", alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, fontsize=7)
axes[1].legend(); axes[1].set_title("Biais vs Variance")
plt.suptitle("Q4 — Analyse Biais/Variance", fontweight="bold")
plt.tight_layout()
path = FIGS_DIR / "q4_bias_variance.png"
fig.savefig(path, dpi=150); plt.show()


### Reponse Q4

| Cas | Configuration | Observation |
|-----|--------------|-------------|
| **Overfitting** | n=10, max_depth=None | Train≈1.0, Test plus faible — peu d'arbres sans contrainte |
| **Underfitting** | n=10, max_depth=2 | Biais eleve — arbres trop peu profonds |
| **Equilibre** | n=100, max_depth=5 a 10 | Meilleur compromis biais-variance |

Augmenter `n_estimators` reduit la **variance**. Augmenter `max_depth` reduit le **biais** mais peut augmenter la variance.


---
## Q5 — Comparaison : Random Forest vs Arbre de Decision vs Autres Modeles


In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models_unscaled = {
    "RandomForest(n=100,d=None)":  RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42),
    "RandomForest(n=100,d=5)":     RandomForestClassifier(n_estimators=100, max_depth=5,    random_state=42),
    "RandomForest(n=200,d=None)":  RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42),
    "DecisionTree(d=None)":        DecisionTreeClassifier(max_depth=None, random_state=42),
    "DecisionTree(d=5)":           DecisionTreeClassifier(max_depth=5,    random_state=42),
    "DecisionTree(d=3)":           DecisionTreeClassifier(max_depth=3,    random_state=42),
    "GradientBoosting":            GradientBoostingClassifier(n_estimators=100, random_state=42),
}
models_scaled = {
    "KNN(k=5)":            KNeighborsClassifier(n_neighbors=5),
    "SVM(rbf)":            SVC(kernel="rbf", C=1.0),
    "LogisticRegression":  LogisticRegression(C=1.0, max_iter=1000),
}

cmp_rows = []
with mlflow.start_run(run_name="Q5_Model_Comparison"):
    for name, model in models_unscaled.items():
        model.fit(X_train, y_train)
        yp_tr = model.predict(X_train); yp = model.predict(X_test)
        cmp_rows.append({"Modele": name,
            "Train Acc": round(accuracy_score(y_train, yp_tr), 4),
            "Test Acc":  round(accuracy_score(y_test,  yp),    4),
            "F1":        round(f1_score(y_test, yp, average="weighted"), 4),
            "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
            "Recall":    round(recall_score(y_test, yp, average="weighted", zero_division=0), 4)})

    for name, model in models_scaled.items():
        model.fit(X_train_sc, y_train)
        yp_tr = model.predict(X_train_sc); yp = model.predict(X_test_sc)
        cmp_rows.append({"Modele": name,
            "Train Acc": round(accuracy_score(y_train, yp_tr), 4),
            "Test Acc":  round(accuracy_score(y_test,  yp),    4),
            "F1":        round(f1_score(y_test, yp, average="weighted"), 4),
            "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
            "Recall":    round(recall_score(y_test, yp, average="weighted", zero_division=0), 4)})
        
    df_cmp = pd.DataFrame(cmp_rows).sort_values("Test Acc", ascending=False)
    df_cmp.to_csv(FIGS_DIR / "q5_comparison.csv", index=False)
    for _, row in df_cmp.iterrows():
        nm = row["Modele"].replace(" ","_").replace("(","").replace(")","").replace("=","").replace(",","")
        mlflow.log_metrics({f"{nm}_train": row["Train Acc"], f"{nm}_test": row["Test Acc"]})

display(df_cmp.reset_index(drop=True))


In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(df_cmp["Modele"], df_cmp["Train Acc"], color="steelblue", alpha=0.5, label="Train")
axes[0].barh(df_cmp["Modele"], df_cmp["Test Acc"],  color="tomato",    alpha=0.85, label="Test")
axes[0].set_xlabel("Accuracy"); axes[0].set_title("Accuracy Train vs Test")
axes[0].legend(); axes[0].set_xlim(0.75, 1.04)
axes[0].axvline(0.95, color="purple", linestyle="--", alpha=0.5)
axes[0].invert_yaxis()

x = np.arange(len(df_cmp)); w = 0.2
axes[1].bar(x - w*1.5, df_cmp["Test Acc"],  w, label="Test Acc",  color="tomato")
axes[1].bar(x - w*0.5, df_cmp["F1"],        w, label="F1",        color="steelblue")
axes[1].bar(x + w*0.5, df_cmp["Precision"], w, label="Precision", color="green")
axes[1].bar(x + w*1.5, df_cmp["Recall"],    w, label="Recall",    color="orange")
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_cmp["Modele"], rotation=45, ha="right", fontsize=7)
axes[1].set_title("Metriques par Modele"); axes[1].set_ylim(0.75, 1.05)
axes[1].legend(fontsize=8)

plt.suptitle("Q5 — Comparaison Multi-Modeles", fontsize=13, fontweight="bold")
plt.tight_layout()
path = FIGS_DIR / "q5_comparison.png"
fig.savefig(path, dpi=150); plt.show()


### Reponse Q5

| Rang | Modele | Interpretation |
|------|--------|---------------|
| 1 | Random Forest (n=200, d=None) | Meilleur generaliste — bagging reduit la variance |
| 2 | Gradient Boosting | Tres performant, boosting sequentiel |
| 3 | SVM / Logistic Regression | Bons sur donnees lineairement separables |
| 4 | Decision Tree (d=None) | Overfitting visible (Train=1.0) |
| 5 | KNN | Raisonnable mais sensible au bruit |

**RF vs Decision Tree :** Le RF surpasse le DT grace au bagging. La moyenne de 100-200 arbres reduit la variance sans augmenter le biais.


---
## Conclusion Generale

### Compromis Biais-Variance dans le Random Forest
- **Bagging** : chaque arbre entraine sur un bootstrap → les erreurs se compensent → variance reduite.
- **Feature sampling** : sqrt(p) features par split → arbres decoreles → variance encore reduite.

### Avantages / Limites

| Avantages | Limites |
|-----------|---------|
| Robuste aux outliers, pas de normalisation | Boite noire, peu interpretable |
| Fournit `feature_importances_` | Lent sur grands datasets |
| Gere les relations non-lineaires | Ne peut pas extrapoler |

> Pour voir toutes les experiences MLflow :
> ```
> mlflow ui --backend-store-uri sqlite:///mlflow.db
> ```
